# 01 — Coleta Atlas (GET) — documentação da disciplina

**Projeto:** Preditor de Falhas ML (Grupo 16)
**Disciplinas:** ED2 + Redes + APS

> Este notebook é a **documentação / demonstração** pedida pela professora.
> O **core** do collector fica no pacote Python (`fetch_measurement_results`).
> A Lambda (S1.7) reutiliza a mesma função — não o notebook.
> **Não** criamos medição aqui (sem POST). Só lemos results de um `msm_id` já existente.


## Equipe

| Integrante | Papel nesta entrega |
|---|---|
| Guilherme Leite Tavares | |
| Alexandre Tiago de Oliveira | |
| Ingrid Ferreira de Sousa | |
| Kauan Garcia Dias de Oliveira | |
| Lucas Eduardo Malachias Bagatela | |
| Stephanie Vitoria Bessa dos Santos | |


## Decisões (justificativa)

| Decisão | Escolha | Por quê |
|---|---|---|
| Verbo HTTP | **GET** `/measurements/{msm_id}/results/` | Collector só lê; POST (criar medição) é S1.6 |
| Onde está o código | `src/preditor_de_falhas_ml/atlas.py` | Reuso pela CLI e pela Lambda |
| Papel deste notebook | Documentação + demo pontual | Exigência da disciplina; não é o pipeline AWS |
| Janela | `start` / `stop` Unix UTC (curta) | Smoke controlado |
| Persistência local | opcional JSONL em `data/raw/` | S3 curated vem na S1.7 |
| Formato na demo | DataFrame pandas (bruto) | Sem agregação / sem `status_real` neste passo |


## Setup

Na raiz do repo: `uv sync`.
Defina `RIPE_ATLAS_API_KEY` no ambiente (ou `.env` — **não** commitado).
Escolha um `MEASUREMENT_ID` **já existente** (público ou criado no S1.6).


In [ ]:
from __future__ import annotations

import json
import os
from datetime import datetime, timezone
from pathlib import Path

from preditor_de_falhas_ml import append_data, fetch_measurement_results

# --- params da demo (ajuste) ---
MEASUREMENT_ID = int(os.environ.get("DEMO_MSM_ID", "0"))  # troque por um msm_id real
HORAS_COLETA = 2
GRAVAR_RAW = True
OUTPUT_DIR = Path("data/raw")

api_key = os.environ.get("RIPE_ATLAS_API_KEY", "")
stop = int(datetime.now(tz=timezone.utc).timestamp())
start = stop - HORAS_COLETA * 3600

{"MEASUREMENT_ID": MEASUREMENT_ID, "start": start, "stop": stop, "HORAS_COLETA": HORAS_COLETA}


## Chamada ao core (GET)

A célula abaixo **importa** a função do pacote. Não usa `requests` direto.


In [ ]:
if not api_key:
    raise RuntimeError("Defina RIPE_ATLAS_API_KEY no ambiente antes de rodar a demo.")
if MEASUREMENT_ID <= 0:
    raise RuntimeError("Defina DEMO_MSM_ID ou edite MEASUREMENT_ID com um id real.")

df_raw = fetch_measurement_results(
    api_key,
    MEASUREMENT_ID,
    start=start,
    stop=stop,
)

assert not df_raw.empty, "GET retornou 0 linhas — confira msm_id e a janela start/stop."
df_raw.head()


In [ ]:
df_raw.info()
len(df_raw)


## Persistência local (opcional)

Só para evidência da disciplina. Produção grava no S3 (S1.7).


In [ ]:
meta = {
    "msm_id": MEASUREMENT_ID,
    "start": start,
    "stop": stop,
    "utc_run": datetime.now(tz=timezone.utc).isoformat(),
    "n_rows": int(len(df_raw)),
    "columns": list(map(str, df_raw.columns)),
    "note": "demo notebook; core = fetch_measurement_results",
}

if GRAVAR_RAW:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now(tz=timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    meta_path = OUTPUT_DIR / f"ripe_atlas_m{MEASUREMENT_ID}_{stamp}_metadata.json"
    meta_path.write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
    out = append_data(df_raw, output_dir=OUTPUT_DIR)
    {
        "jsonl": str(out),
        "metadata": str(meta_path),
        "bytes_jsonl": out.stat().st_size if out.exists() else 0,
    }
else:
    meta


## Checklist

- [ ] Equipe preenchida
- [ ] Decisões justificadas
- [ ] GET via `fetch_measurement_results` (sem POST / sem `requests` nas células)
- [ ] DataFrame exibido sem agregação
- [ ] (Opcional) raw + metadata em `data/raw/`
- [ ] Claro para a banca: notebook = doc; produção = pacote + Lambda
